In [1]:
import os
BASE = "/kaggle/input/datasets/pramcharanteja/svamitva-14class-unified-train"
for village in os.listdir(BASE):
    full = f"{BASE}/{village}"
    if not os.path.isdir(full): continue
    imgs = len(os.listdir(f"{full}/images")) if os.path.exists(f"{full}/images") else 0
    lbls = len(os.listdir(f"{full}/labels")) if os.path.exists(f"{full}/labels") else 0
    print(f"{'✅' if imgs and lbls else '⚠️'}  {village}: {imgs} img, {lbls} lbl")

✅  chhattisgarh_murdanda_unified_vol2: 2870 img, 550 lbl
✅  punjab_fattu_bhila_vol1: 1120 img, 581 lbl
✅  chhattisgarh_samlur_unified_vol1: 1437 img, 290 lbl
✅  chhattisgarh_murdanda_unified_vol3: 250 img, 55 lbl
✅  punjab_bagga_vol1: 1435 img, 720 lbl
✅  chhattisgarh_badetumnar_unified_vol2: 710 img, 132 lbl
✅  chhattisgarh_murdanda_unified_vol1: 2870 img, 549 lbl
✅  chhattisgarh_badetumnar_unified_vol1: 2870 img, 543 lbl
✅  chhattisgarh_nagul_unified_vol1: 1701 img, 330 lbl


In [2]:
import shutil
WORKING = "/kaggle/working"
for folder in ["data", "labels_fixed", "runs"]:
    shutil.rmtree(f"{WORKING}/{folder}", ignore_errors=True)
import shutil as sh
_, _, free = sh.disk_usage(WORKING)
print(f"Free: {free//1e9:.1f}GB")

Free: 20.0GB


In [3]:
import os
BASE    = "/kaggle/input/datasets/pramcharanteja/svamitva-14class-unified-train"
WORKING = "/kaggle/working"
VILLAGE_SPLIT = {
    "chhattisgarh_samlur_unified_vol1":     "train",
    "chhattisgarh_murdanda_unified_vol1":   "train",
    "chhattisgarh_murdanda_unified_vol2":   "train",
    "chhattisgarh_murdanda_unified_vol3":   "train",
    "chhattisgarh_nagul_unified_vol1":      "train",
    "punjab_fattu_bhila_vol1":              "train",
    "chhattisgarh_badetumnar_unified_vol1": "val",
    "chhattisgarh_badetumnar_unified_vol2": "val",
    "punjab_bagga_vol1":                    "val",
}
INFLATION = {6: 0.050, 7: 0.060, 8: 0.025, 14: 0.025}
REMAP     = {0:0, 1:1, 2:2, 3:3, 4:4, 5:5, 6:6, 7:7, 13:8, 14:9}

for split in ["train", "val"]:
    os.makedirs(f"{WORKING}/labels_fixed/{split}", exist_ok=True)

written = 0
for village, split in VILLAGE_SPLIT.items():
    src = f"{BASE}/{village}/labels"
    dst = f"{WORKING}/labels_fixed/{split}"
    if not os.path.exists(src): continue
    for fname in os.listdir(src):
        lines = open(f"{src}/{fname}").readlines()
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if len(parts) != 5: continue
            cls, cx, cy, w, h = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            if cls not in REMAP: continue
            if cls in INFLATION:
                min_s = INFLATION[cls]
                w = max(w, min_s); h = max(h, min_s)
                cx = min(max(cx, w/2), 1-w/2)
                cy = min(max(cy, h/2), 1-h/2)
            new_lines.append(f"{REMAP[cls]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")
        open(f"{dst}/{fname}", "w").writelines(new_lines)
        written += 1

print(f"Written {written} label files")
for split in ["train", "val"]:
    print(f"labels_fixed/{split}: {len(os.listdir(f'{WORKING}/labels_fixed/{split}'))} ✅")

Written 3750 label files
labels_fixed/train: 2355 ✅
labels_fixed/val: 1395 ✅


In [4]:
from PIL import Image
import os, shutil

BASE    = "/kaggle/input/datasets/pramcharanteja/svamitva-14class-unified-train"
WORKING = "/kaggle/working"
VILLAGE_SPLIT = {
    "chhattisgarh_samlur_unified_vol1":     "train",
    "chhattisgarh_murdanda_unified_vol1":   "train",
    "chhattisgarh_murdanda_unified_vol2":   "train",
    "chhattisgarh_murdanda_unified_vol3":   "train",
    "chhattisgarh_nagul_unified_vol1":      "train",
    "punjab_fattu_bhila_vol1":              "train",
    "chhattisgarh_badetumnar_unified_vol1": "val",
    "chhattisgarh_badetumnar_unified_vol2": "val",
    "punjab_bagga_vol1":                    "val",
}

for split in ["train", "val"]:
    os.makedirs(f"{WORKING}/data/images/{split}", exist_ok=True)

for village, split in VILLAGE_SPLIT.items():
    src = f"{BASE}/{village}/images"
    dst = f"{WORKING}/data/images/{split}"
    if not os.path.exists(src): continue
    files = os.listdir(src)
    for i, fname in enumerate(files):
        out = f"{dst}/{os.path.splitext(fname)[0]}.jpg"
        if os.path.exists(out): continue
        img = Image.open(f"{src}/{fname}").convert("RGB")
        img.save(out, "JPEG", quality=95)
        if i % 1000 == 0: print(f"{village} {split}: {i}/{len(files)}")
    print(f"✅ {village}: {len(os.listdir(dst))} jpgs")

_, _, free = shutil.disk_usage(WORKING)
print(f"\nFree after JPG conversion: {free//1e9:.1f}GB")

chhattisgarh_samlur_unified_vol1 train: 0/1437
chhattisgarh_samlur_unified_vol1 train: 1000/1437
✅ chhattisgarh_samlur_unified_vol1: 1437 jpgs
chhattisgarh_murdanda_unified_vol1 train: 0/2870
chhattisgarh_murdanda_unified_vol1 train: 1000/2870
chhattisgarh_murdanda_unified_vol1 train: 2000/2870
✅ chhattisgarh_murdanda_unified_vol1: 4273 jpgs
chhattisgarh_murdanda_unified_vol2 train: 0/2870
chhattisgarh_murdanda_unified_vol2 train: 1000/2870
chhattisgarh_murdanda_unified_vol2 train: 2000/2870
✅ chhattisgarh_murdanda_unified_vol2: 7108 jpgs
chhattisgarh_murdanda_unified_vol3 train: 0/250
✅ chhattisgarh_murdanda_unified_vol3: 7355 jpgs
chhattisgarh_nagul_unified_vol1 train: 1000/1701
✅ chhattisgarh_nagul_unified_vol1: 8882 jpgs
punjab_fattu_bhila_vol1 train: 0/1120
punjab_fattu_bhila_vol1 train: 1000/1120
✅ punjab_fattu_bhila_vol1: 9865 jpgs
chhattisgarh_badetumnar_unified_vol1 val: 0/2870
chhattisgarh_badetumnar_unified_vol1 val: 1000/2870
chhattisgarh_badetumnar_unified_vol1 val: 2000/2

In [5]:
import os, shutil

WORKING = "/kaggle/working"

# YOLO finds labels by replacing 'images' with 'labels' in the path.
# images at: data/images/train  →  labels at: data/labels/train ✅
for split in ["train", "val"]:
    dst = f"{WORKING}/data/labels/{split}"
    if os.path.islink(dst): os.unlink(dst)
    shutil.rmtree(dst, ignore_errors=True)
    os.makedirs(f"{WORKING}/data/labels", exist_ok=True)
    os.symlink(f"{WORKING}/labels_fixed/{split}", dst)
    print(f"data/labels/{split}: {len(os.listdir(dst))} files ✅")

data/labels/train: 2355 files ✅
data/labels/val: 1395 files ✅


In [6]:
import yaml
WORKING = "/kaggle/working"

data = {
    "path": WORKING,
    "train": "data/images/train",
    "val":   "data/images/val",
    "nc": 10,
    "names": {
        0:"Building_RCC", 1:"Building_Tiled", 2:"Building_Tin", 3:"Building_Other",
        4:"Road_Polygon", 5:"Waterbody_Polygon", 6:"Transformer", 7:"Overhead_Tank",
        8:"Utility_Polygon", 9:"Waterbody_Point"
    }
}
with open(f"{WORKING}/data.yaml", "w") as f:
    yaml.dump(data, f)
print(open(f"{WORKING}/data.yaml").read())

names:
  0: Building_RCC
  1: Building_Tiled
  2: Building_Tin
  3: Building_Other
  4: Road_Polygon
  5: Waterbody_Polygon
  6: Transformer
  7: Overhead_Tank
  8: Utility_Polygon
  9: Waterbody_Point
nc: 10
path: /kaggle/working
train: data/images/train
val: data/images/val



In [7]:
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.3 MB/s eta 0:00:00


In [8]:
from ultralytics import YOLO
import os

WORKING    = "/kaggle/working"
CHECKPOINT = f"{WORKING}/runs/svamitva_v1/weights/last.pt"

if os.path.exists(CHECKPOINT):
    print(f"Resuming: {CHECKPOINT}")
    model = YOLO(CHECKPOINT)
    model.train(
        data=f"{WORKING}/data.yaml",
        epochs=100, imgsz=640, batch=16, device=0,
        project=f"{WORKING}/runs", name="svamitva_v1",
        resume=True, save=True, save_period=5,
        workers=2, cache=False, exist_ok=True,
    )
else:
    print("Starting fresh")
    model = YOLO("/kaggle/input/models/ultralytics/yolo11/pytorch/default/1/yolo11s.pt")
    model.train(
        data=f"{WORKING}/data.yaml",
        epochs=100, imgsz=640, batch=16, device=0,
        project=f"{WORKING}/runs", name="svamitva_v1",
        save=True, save_period=5,
        copy_paste=0.5, mosaic=1.0, degrees=45.0,
        fliplr=0.5, flipud=0.5, scale=0.5,
        hsv_h=0.015, hsv_s=0.4, hsv_v=0.35,
        patience=20, lr0=0.01, warmup_epochs=3,
        workers=2, cache=False,
    )

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Starting fresh
Ultralytics 8.4.23 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.5, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=45.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.4, hsv_v=0.35, img

In [9]:
print("hi")

hi
